# 🧬 Bio-AI: Predicting VHL Mutation & ccRCC from Histopathology Slides
### AI-Driven Deep Learning & Morphological Explainability with Grad-CAM

---

## 🔬 Biological Background
- **VHL (Von Hippel-Lindau)** is a classical tumor suppressor gene located on chromosome 3p25.3.
- In normal cells, **pVHL** acts as the recognition component of an E3 ubiquitin ligase complex that targets **HIF-1α / HIF-2α** for proteasomal degradation under normoxia.
- **Loss-of-function mutations** of VHL occur in **70–80% of Clear Cell Renal Cell Carcinomas (ccRCC)**.
- Without functional pVHL, HIF accumulates and drives:
  - **Angiogenesis:** Hypersecretion of VEGF, creating delicate arborizing capillary networks.
  - **Metabolic Alterations:** Overexpression of GLUT1/SLC2A1 and glycogen/lipid accumulation, resulting in the optical **clear cytoplasm** hallmark.

In this project, we train a Deep Convolutional Neural Network (Transfer Learning with **ResNet18**) to identify the VHL mutation phenotype from H&E histopathology patches and use **Grad-CAM** to verify the morphological decision drivers.

### Step 1: GPU & Environment Setup

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Step 2: Install and Import Packages

In [ ]:
!pip install -q opencv-python seaborn scikit-learn albumentations

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, ImageDraw

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

### Step 3: Histopathology Benchmark Dataset Generation

In [ ]:
def generate_he_patch(is_vhl_mutant: bool, size=(224, 224)) -> Image.Image:
    img_array = np.zeros((size[0], size[1], 3), dtype=np.uint8)
    if is_vhl_mutant:
        base_color = np.array([238, 225, 230], dtype=np.float32)
        noise = np.random.normal(0, 10, img_array.shape).astype(np.float32)
        img = Image.fromarray(np.clip(base_color + noise, 0, 255).astype(np.uint8))
        draw = ImageDraw.Draw(img)
        for _ in range(35):
            x, y = np.random.randint(10, size[0]-20), np.random.randint(10, size[1]-20)
            r = np.random.randint(8, 20)
            draw.ellipse([x-r, y-r, x+r, y+r], fill=(250, 248, 250), outline=(210, 180, 195))
        for _ in range(40):
            nx, ny = np.random.randint(5, size[0]-10), np.random.randint(5, size[1]-10)
            nr = np.random.randint(3, 7)
            draw.ellipse([nx-nr, ny-nr, nx+nr, ny+nr], fill=(70, 25, 95))
        for _ in range(6):
            pts = [(np.random.randint(0, size[0]), np.random.randint(0, size[1])) for _ in range(4)]
            draw.line(pts, fill=(180, 50, 70), width=1)
    else:
        base_color = np.array([215, 150, 185], dtype=np.float32)
        noise = np.random.normal(0, 12, img_array.shape).astype(np.float32)
        img = Image.fromarray(np.clip(base_color + noise, 0, 255).astype(np.uint8))
        draw = ImageDraw.Draw(img)
        for _ in range(55):
            nx, ny = np.random.randint(5, size[0]-10), np.random.randint(5, size[1]-10)
            nr = np.random.randint(2, 5)
            draw.ellipse([nx-nr, ny-nr, nx+nr, ny+nr], fill=(60, 20, 85))
    return img

DATA_DIR = Path("sample_data")
for cname in ["VHL_WildType", "VHL_Mutant"]:
    (DATA_DIR / cname).mkdir(parents=True, exist_ok=True)

print("Generating sample histopathology patches...")
for i in range(75):
    generate_he_patch(is_vhl_mutant=False).save(DATA_DIR / "VHL_WildType" / f"wt_{i:03d}.png")
    generate_he_patch(is_vhl_mutant=True).save(DATA_DIR / "VHL_Mutant" / f"mut_{i:03d}.png")
print("✅ 150 patches generated.")

### Step 4: Visualizing Morphological Differences

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(DATA_DIR / "VHL_WildType" / "wt_000.png"))
axes[0].set_title("VHL Wild-Type (Tubular / Eosinophilic)", fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(Image.open(DATA_DIR / "VHL_Mutant" / "mut_000.png"))
axes[1].set_title("VHL Mutant (ccRCC Clear Cell & Capillaries)", fontsize=12, fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### Step 5: Dataset & Transforms

In [ ]:
class HistoDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=(0, 180)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

paths, labels = [], []
for idx, cname in enumerate(["VHL_WildType", "VHL_Mutant"]):
    for p in (DATA_DIR / cname).glob("*.png"):
        paths.append(p)
        labels.append(idx)

train_p, test_p, train_l, test_l = train_test_split(paths, labels, test_size=0.2, stratify=labels, random_state=42)
train_p, val_p, train_l, val_l = train_test_split(train_p, train_l, test_size=0.15, stratify=train_l, random_state=42)

train_loader = DataLoader(HistoDataset(train_p, train_l, train_tf), batch_size=16, shuffle=True)
val_loader = DataLoader(HistoDataset(val_p, val_l, val_tf), batch_size=16, shuffle=False)
test_loader = DataLoader(HistoDataset(test_p, test_l, val_tf), batch_size=16, shuffle=False)
print(f"Dataset: Train={len(train_p)}, Val={len(val_p)}, Test={len(test_p)}")

### Step 6: Model Architecture & Training

In [ ]:
class VHLClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_f = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.target_conv = self.backbone.layer4[-1]
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_f, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

model = VHLClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 8
for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss = 0.0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), targets)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
    t_loss /= len(train_loader.dataset)

    model.eval()
    v_loss, correct = 0.0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            out = model(imgs)
            v_loss += criterion(out, targets).item() * imgs.size(0)
            correct += (torch.argmax(out, dim=1) == targets).sum().item()
    v_loss /= len(val_loader.dataset)
    v_acc = correct / len(val_loader.dataset)
    print(f"Epoch [{epoch:02d}/{EPOCHS}] | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Acc: {v_acc*100:.1f}%")

### Step 7: Test Evaluation (ROC & Confusion Matrix)

In [ ]:
model.eval()
all_targets, all_preds, all_probs = [], [], []
with torch.no_grad():
    for imgs, targets in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        all_targets.extend(targets.numpy())
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())

print(classification_report(all_targets, all_preds, target_names=["VHL_WildType", "VHL_Mutant"]))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=["VHL_WildType", "VHL_Mutant"], yticklabels=["VHL_WildType", "VHL_Mutant"])
axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(all_targets, all_probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='red', label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

### Step 8: Grad-CAM Explainability

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations, self.gradients = None, None
        self.target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        self.target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))
    def generate(self, input_tensor, target_class=1):
        self.model.eval()
        logits = self.model(input_tensor)
        self.model.zero_grad()
        logits[0, target_class].backward()
        weights = torch.mean(self.gradients[0], dim=(1, 2), keepdim=True)
        cam = torch.sum(weights * self.activations[0], dim=0)
        cam = torch.clamp(cam, min=0).cpu().numpy()
        if cam.max() > 0:
            cam /= cam.max()
        return cam

sample_patch = DATA_DIR / "VHL_Mutant" / "mut_001.png"
raw_img = Image.open(sample_patch).convert("RGB")
inp_tensor = val_tf(raw_img).unsqueeze(0).to(device)

gradcam = GradCAM(model, model.target_conv)
cam = gradcam.generate(inp_tensor, target_class=1)

raw_np = np.array(raw_img.resize((224, 224)), dtype=np.float32) / 255.0
cam_resized = cv2.resize(cam, (224, 224))
heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
overlay = 0.5 * raw_np + 0.5 * heatmap

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(raw_np)
axes[0].set_title("Original H&E Tissue")
axes[0].axis('off')
axes[1].imshow(cam, cmap='jet')
axes[1].set_title("Grad-CAM Heatmap")
axes[1].axis('off')
axes[2].imshow(overlay)
axes[2].set_title("Clear Cytoplasm Focus")
axes[2].axis('off')
plt.tight_layout()
plt.show()